# Part 14, notebook 4 of 6: how much should a sub-agent be given?

**The picture.** You ask an assistant to read one report and tell you two facts. You can hand them just that report, or that report buried in a stack of 19, or in a stack of 99. Does a bigger stack make them sloppier ("context rot": studying with the TV on)?

The test: same analyst prompt, same question about **one** report (its verdict and the confidence of its first kept candidate), three brief sizes, with the target at the start, middle or end of the padded briefs. 12 targets, 84 runs, exact scoring. This notebook reads the saved 84 runs and then makes **2 live calls** to show one case end to end.

In [1]:
import sys, json, os
from pathlib import Path
os.environ.setdefault("AGENTIC_ZERO_MODEL", "qwen3.8-27b")   # the model name the local vLLM server serves (as in the post)
os.environ.setdefault("RUST_LOG", "error"); os.environ.setdefault("LANCE_LOG", "error")   # silence LanceDB deprecation chatter
import warnings; warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("..").resolve()))                # this notebook lives in literate/, the code in notebooks/agentic-zero-to-advanced/
OUT = Path("..").resolve() / "out"
import ctx_subagent as sa
runs = json.loads((OUT / "ctx_subagent_runs.json").read_text())
print(len(runs), "saved runs;", len({r["target"] for r in runs}), "target reports")

84 saved runs; 12 target reports


## The results

In [2]:
from collections import defaultdict
g = defaultdict(list)
for r in runs: g[r["arm"]].append(r)
order = ["narrow", "pad20-start", "pad20-middle", "pad20-end", "pad100-start", "pad100-middle", "pad100-end"]
print(f"{'brief':14s} {'correct':>8s} {'mean prompt tokens':>19s}")
for a in order:
    rs = g[a]; print(f"{a:14s} {sum(r['correct'] for r in rs):>4d}/{len(rs):<3d} {sum(r['prompt_tokens'] for r in rs)/len(rs):19.0f}")
narrow = sum(r["prompt_tokens"] for r in g["narrow"]) / 12; big = sum(r["prompt_tokens"] for r in g["pad100-middle"]) / 12
print(f"\nthe narrow brief uses {big / narrow:.0f}x fewer tokens ({narrow:.0f} against {big:.0f})")

brief           correct  mean prompt tokens
narrow           12/12                  364
pad20-start      12/12                 2946
pad20-middle     12/12                 2946
pad20-end        12/12                 2946
pad100-start     12/12                13924
pad100-middle    12/12                13924
pad100-end       12/12                13924

the narrow brief uses 38x fewer tokens (364 against 13924)


**No context rot showed up, at this scale and with this model.** All 84 runs were exactly right, including with the target in the middle of 100 reports. What the narrow brief bought was cost: about 38 times fewer tokens, and with them time, money and a smaller surface for injected text.

That is **not** evidence that rot does not exist. 14,000 tokens is not enough to trigger it for a 27B model on a task whose answer is two exact lines. Published rot results are at much longer lengths and on fuzzier questions.

## One case, live

The brief is just files pasted under a header. `arms(target)` builds all seven for a target. Here the narrow brief and the 100-report brief with the target in the middle, both sent to the model.

In [3]:
import time
from ctx_harness import client, MODEL, NO_THINK, ANALYST_SYSTEM
tid = sa.target_ids()[0]
gold = sa.gold(tid)
q = f"Question: for report {tid}, what is the verdict and the confidence of the FIRST kept candidate listed?"
arms = dict(sa.arms(tid))
print("target:", tid, "| gold:", gold, "\n")
for arm in ("narrow", "pad100-middle"):
    body = sa.brief(arms[arm]); t0 = time.time()
    r = client.chat.completions.create(model=MODEL, temperature=0, max_tokens=200, extra_body=NO_THINK,
        messages=[{"role": "system", "content": ANALYST_SYSTEM + " " + sa.FORMAT}, {"role": "user", "content": f"{q}\n\n{body}"}])
    print(f"{arm:14s} files={len(arms[arm]):3d} prompt tokens={r.usage.prompt_tokens:6d} ({time.time()-t0:.1f}s) -> {r.choices[0].message.content.strip()}")

target: test_00072 | gold: {'verdict': 'escalate', 'confidence': 0.4} 



narrow         files=  1 prompt tokens=   377 (1.5s) -> ANSWER: {"verdict": "escalate", "confidence": 0.4}


pad100-middle  files=100 prompt tokens= 13799 (8.6s) -> - Report test_00072 has a verdict of "escalate".
- The first kept candidate listed is candidate 0.
- Candidate 0 has a confidence of 0.4.
- Candidate 0 is marked as uncertain (uncertain true).

source: reports/test_00072.md
ANSWER: {"verdict": "escalate", "confidence": 0.4}


Both get it right; one costs a fraction. I would still give a sub-agent the narrow brief, because it is free and it is the version that scales, but this experiment gives you a reason of **cost**, not of accuracy.